# Semana 9 · Sesión 2: Espín y oscilador armónico

**Módulo 2**

## Objetivos de la sesión

1. Usar `physics.vector` para calcular divergencia, rotacional y gradiente de
   un campo, y reconocer un campo conservativo.
2. Trabajar con espín y momento angular en `quantum.spin`: aplicar operadores
   con `qapply`, obtener matrices con `represent` y combinar espines con
   `TensorProduct`.
3. Resolver el oscilador armónico con operadores de escalera, sin escribir
   ninguna función de onda.

## Retomamos

La sesión pasada los operadores eran abstractos: `A`, `B`, `C`, sin
representación. Hoy son concretos: $J_x, J_y, J_z$ actúan sobre estados
$|j, m\rangle$ y `qapply` calcula el resultado. Antes, un repaso breve de
`physics.vector`, que vimos con `mechanics` la semana 7, aplicado a campos.

Esta celda es el único preparativo: no depende de la sesión anterior.

In [ ]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math
from sympy.physics.matrices import msigma
from sympy.physics.quantum import Commutator, Dagger, TensorProduct, qapply, represent
from sympy.physics.quantum.sho1d import LoweringOp, NumberOp, RaisingOp, SHOBra, SHOKet
from sympy.physics.quantum.spin import J2, Jminus, Jplus, Jx, Jy, Jz, JzKet

me.init_vprinting()

N = me.ReferenceFrame("N")
x, y, z = N[0], N[1], N[2]       # las coordenadas del marco N

## Campos con `physics.vector`

En un marco de referencia, las funciones `me.divergence`, `me.curl` y
`me.gradient` operan sobre campos escritos con las coordenadas `N[0]`,
`N[1]` y `N[2]`. Un `Vector` no es una expresión de SymPy: se muestra con
`Math(me.vlatex(...))`.

Dos campos para comparar: un campo central tipo Coulomb, y una rotación rígida.

In [ ]:
radio = sp.sqrt(x**2 + y**2 + z**2)
campo_coulomb = (x * N.x + y * N.y + z * N.z) / radio**3
campo_rotacion = -y * N.x + x * N.y

display(sp.Eq(sp.Symbol("div_E"), sp.simplify(me.divergence(campo_coulomb, N))))
display(Math(r"\nabla\times\mathbf{E} = " + me.vlatex(me.curl(campo_coulomb, N))))
display(Math(r"\nabla\times\mathbf{v} = " + me.vlatex(me.curl(campo_rotacion, N))))

El campo de Coulomb tiene rotacional cero (es conservativo, deriva de un
potencial) y divergencia cero fuera del origen; la rotación rígida tiene
rotacional $2\hat{\mathbf z}$. El gradiente recupera el campo desde el
potencial $\phi = -1/r$.

In [ ]:
potencial = -1 / radio

display(Math(r"\nabla\phi = " + me.vlatex(me.gradient(potencial, N))))
display(sp.simplify((me.gradient(potencial, N) - campo_coulomb).to_matrix(N)))

## Espín: estados y operadores

`JzKet(j, m)` es el estado $|j, m\rangle$. Los operadores `Jx`, `Jy`, `Jz`,
`J2`, `Jplus` y `Jminus` ya traen sus reglas, con $\hbar$ como símbolo, y
`qapply` las aplica. Para $j = 1$:

In [ ]:
estado = JzKet(1, 0)

display(sp.Eq(Jz * estado, qapply(Jz * estado), evaluate=False))
display(sp.Eq(J2 * estado, qapply(J2 * estado), evaluate=False))
display(sp.Eq(Jplus * estado, qapply(Jplus * estado), evaluate=False))

Observa que $J_+$ sube $m$, que $J_z|1,0\rangle = 0$ y que
$J^2 = \hbar^2 j(j+1) = 2\hbar^2$. Los operadores de Cartesianas cumplen el
álgebra de momento angular, que `Commutator` reproduce sola:

In [ ]:
display(sp.Eq(Commutator(Jx, Jy), Commutator(Jx, Jy).doit(), evaluate=False))
display(sp.Eq(Commutator(J2, Jz), Commutator(J2, Jz).doit(), evaluate=False))

## De operadores a matrices: `represent`

`represent(operador, basis=Jz, j=...)` da la matriz del operador en la base
$|j, m\rangle$. Para $j = 1/2$ salen las matrices de Pauli por $\hbar/2$:

In [ ]:
semi = sp.Rational(1, 2)
hbar = sp.Symbol("hbar", positive=True)  # el mismo hbar de quantum.spin

matriz_jx = represent(Jx, basis=Jz, j=semi)
matriz_jy = represent(Jy, basis=Jz, j=semi)

display(Math(r"J_x = " + sp.latex(matriz_jx)))   # una matriz no entra en sp.Eq: Eq la daría por distinta de un símbolo
display(Math(r"J_y = " + sp.latex(matriz_jy)))
display(Math(r"\sigma_x = " + sp.latex(msigma(1))))

## TODO en clase 1

El **espín 1**. Con $j = 1$ las matrices son $3\times 3$.

1. Calcula `jx_1`, `jy_1` y `jz_1` con `represent(..., basis=Jz, j=1)`.
2. Comprueba el álgebra con matrices: `jx_1 * jy_1 - jy_1 * jx_1` debe ser
   $i\hbar J_z$ (resta `sp.I * hbar * jz_1` y simplifica a la matriz cero).
3. Obtén los eigenvalores de `jx_1` con `eigenvals()`. ¿Por qué son los mismos
   que los de $J_z$?
4. Aplica `Jminus` dos veces al estado `JzKet(1, 1)` con `qapply` y guárdalo
   en `estado_abajo`.

In [ ]:
# TODO en clase: el espín 1
jx_1 = ...
jy_1 = ...
jz_1 = ...
conmutador_matrices = ...
valores_propios = ...
estado_abajo = ...

## Dos espines: `TensorProduct`

Para dos partículas de espín 1/2 el espacio es el producto tensorial. Con
matrices se hace con `TensorProduct`. El hamiltoniano de Heisenberg
$H = \vec\sigma_1\cdot\vec\sigma_2$ separa el estado **singlete** (el de
espín total 0) de los tres estados del **triplete**:

In [ ]:
sx, sy, sz = [sp.Matrix(msigma(k)) for k in (1, 2, 3)]

heisenberg = (
    TensorProduct(sx, sx) + TensorProduct(sy, sy) + TensorProduct(sz, sz)
)

display(Math(r"H = " + sp.latex(heisenberg)))
display(heisenberg.eigenvals())

El triplete tiene energía $+1$ (tres veces) y el singlete $-3$. Los
eigenvalores de $\vec\sigma_1\cdot\vec\sigma_2$ son $2S(S+1) - 3$ con $S$ el
espín total, así que $S = 1 \to +1$ y $S = 0 \to -3$.

## El oscilador armónico con operadores

En el módulo `sho1d` los operadores de escalera son `RaisingOp` (el $a^\dagger$)
y `LoweringOp` (el $a$), y los estados $|n\rangle$ son `SHOKet(n)`. Un
operador de subir y uno de bajar con el mismo nombre forman un par:
$[a, a^\dagger] = 1$, y el operador número es $N = a^\dagger a$.

In [ ]:
a = LoweringOp("a")
a_dag = RaisingOp("a")
numero = NumberOp("N")

display(sp.Eq(Commutator(a, a_dag), Commutator(a, a_dag).doit(), evaluate=False))
display(sp.Eq(Commutator(numero, a_dag), Commutator(numero, a_dag).doit(), evaluate=False))

In [ ]:
n = 2   # un estado concreto para ver las reglas

display(sp.Eq(a_dag * SHOKet(n), qapply(a_dag * SHOKet(n)), evaluate=False))
display(sp.Eq(a * SHOKet(n), qapply(a * SHOKet(n)), evaluate=False))
display(sp.Eq(numero * SHOKet(n), qapply(numero * SHOKet(n)), evaluate=False))

$a^\dagger|n\rangle = \sqrt{n+1}\,|n+1\rangle$ y
$a|n\rangle = \sqrt{n}\,|n-1\rangle$. Con eso basta para calcular cualquier
valor esperado: la posición y el momento son combinaciones de $a$ y $a^\dagger$,

$$x = \sqrt{\frac{\hbar}{2m\omega}}\,(a + a^\dagger).$$

Calculemos $\langle n|x^2|n\rangle$ en $n = 2$ expandiendo el cuadrado y
aplicando `qapply` entre un bra y un ket:

In [ ]:
m, omega = sp.symbols("m omega", positive=True)   # masa y frecuencia: positivas

operador_x = sp.sqrt(hbar / (2 * m * omega)) * (a + a_dag)
x_cuadrada = qapply(SHOBra(n) * operador_x * operador_x * SHOKet(n))

display(sp.Eq(sp.Symbol("x2_n2"), x_cuadrada))

## TODO en clase 2

El **principio de incertidumbre** del oscilador, para el estado $|n\rangle$.

El momento es $p = i\sqrt{\dfrac{m\hbar\omega}{2}}\,(a^\dagger - a)$.

1. Define `operador_p` con esa fórmula.
2. Calcula `x2` y `p2`, los valores esperados de $x^2$ y $p^2$ en $|n\rangle$
   con $n = 3$. Usa la misma receta de arriba: `qapply(SHOBra(3) * ... * SHOKet(3))`.
3. Como $\langle x\rangle = \langle p\rangle = 0$, la incertidumbre es
   $\Delta x\,\Delta p = \sqrt{\langle x^2\rangle\langle p^2\rangle}$.
   Guárdala en `incertidumbre` y compárala con $\hbar/2$.
4. ¿Para qué $n$ se alcanza el mínimo de Heisenberg?

In [ ]:
# TODO en clase: la incertidumbre del oscilador
operador_p = ...
x2 = ...
p2 = ...
incertidumbre = ...

## Resumen

Con `physics.vector` el rotacional, la divergencia y el gradiente se calculan
en un marco; un campo central tiene rotacional cero y se recupera de su
potencial.

Con `quantum.spin` los operadores de momento angular actúan sobre
$|j, m\rangle$ vía `qapply`, y `represent` los vuelve matrices: para
$j = 1/2$ son las de Pauli, y `TensorProduct` junta dos espines.
Con `sho1d` el oscilador se resuelve con $a$, $a^\dagger$ y $N$: las reglas de
escalera bastan para obtener valores esperados sin una sola integral.

**Tarea de la semana:** [`tarea/tarea-09.ipynb`](../tarea/tarea-09.ipynb) —
unidades de una constante atómica, un conmutador, un espín en una dirección
arbitraria y el oscilador.

**Próxima semana — Semana 10:** integración de herramientas y proyecto
intermedio: cómo organizar un notebook largo en varios archivos y cómo
revisar el PR de un compañero.